In [1]:
import folium
import pandas as pd
import osmnx as ox
import networkx as nx
import numpy as np

In [2]:
routes = pd.read_csv('../data/solution_routes.csv')
sample = pd.read_csv('../data/sample_stops.csv')
distance_matrix = np.load('../data/distance_matrix.npy').astype(int)
G = ox.load_graphml('../data/raleigh_graph.graphml')
m = folium.Map(location = [35.8, -78.6], zoom_start=11)

In [3]:
print(sample.columns)

Index(['Address', 'City Limits', 'latitude', 'longitude', 'node_id', 'demand'], dtype='str')


In [4]:
colors = ['blue', 'red', 'green', 'purple', 'orange', 'darkred']
depot = routes[routes['stop_sequence'] == 0].iloc[0]
folium.Marker(
    location = [routes.iloc[0]['latitude'], routes.iloc[0]['longitude']],
    popup = 'DRT3 Depot - Amazon',
    icon = folium.Icon(color = 'black', icon = 'star')
).add_to(m)

In [5]:
layers = {}

for vehicle_id in sorted(routes['vehicle'].unique()):
    vr = routes[routes['vehicle'] == vehicle_id].sort_values('stop_sequence')
    color = colors[vehicle_id - 1]
    layer = folium.FeatureGroup(name=f'Driver {vehicle_id}')
    layers[vehicle_id] = layer

    cumulative_distance = 0
    cumulative_time = 0
    prev_node = 0

    for _, row in vr.iterrows():
        current_node = int(row['node'])
        leg = distance_matrix[prev_node][current_node]
        cumulative_distance += leg
        cumulative_time += (leg / 1000) / 30 * 60 + 5      # 5 min service time

        if current_node == 0 and row['stop_sequence'] > 0:
            prev_node = current_node
            continue                                        # skip depot return marker

        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=6,
            color=color,
            fill=True,
            fill_opacity=0.8,
            popup=folium.Popup(f"""
                <b>Driver {vehicle_id} — Stop {int(row['stop_sequence'])}</b><br>
                <b>Address:</b> {row['address']}<br>
                <b>Arrival:</b> {cumulative_time:.0f} min from depot<br>
                <b>Distance:</b> {cumulative_distance/1000:.1f} km<br>
                <b>Packages:</b> {int(sample.iloc[current_node]['demand'])}
            """, max_width=250),
        ).add_to(layer)

        prev_node = current_node

    nodes = vr['node'].tolist()
    failed = 0
    for i in range(len(nodes) - 1):
        try:
            path = nx.shortest_path(
                G,
                int(sample.iloc[nodes[i]]['node_id']),
                int(sample.iloc[nodes[i + 1]]['node_id']),
                weight='length',
            )
            folium.PolyLine(
                [[G.nodes[n]['y'], G.nodes[n]['x']] for n in path],
                color=color, weight=3, opacity=0.7,
            ).add_to(layer)
        except nx.NetworkXNoPath:
            failed += 1

    if failed:
        print(f"Driver {vehicle_id}: {failed} legs could not be drawn")

    layer.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

In [6]:
m.save('../output/route_map.html')